# AERPAW Use Case Demonstration
* This Demonstration showcases the AERPAWEnvironment's specialized funcitonality for use with the AERPAW digial twin
* Includes initialization, visualization, and SNR calculation

## This Environment Provides
* A quick and minimial initialization of the simulation environment
* Straighforward ray-tracing computation of SNR values between all transmitters and receivers in the environment
* All the existing functionality of the general environment

## What's Next
* Incorporating Blender scripting to automatically fetch topology, building, and natural environment data from OpenStreetMaps and OpenTopography for a provided region
* Integrating this functionality directly into the AERPAW environment, to provide an accurate ray-tracing SNR model

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from AERPAWEnvironment import AERPAWEnv

# Defining the scene, including receivers and transmitters

# Defining a generic UAV for simplicity
generic_uav = {
    "device_type": "tx",
    "mass": 5,
    "efficiency": 0.7,
    "position": np.zeros(3),
    "velocity": np.zeros(3),
    "color": np.array([1, 0, 0]),
    "bandwidth": 50,
    "rotor_area": 0.25,
    "signal_power": 3,
    "throughput_capacity": 625000000,
    "battery_capacity": 10000
}

# Configuring some generic uavs with random x and y positions
num_uavs = 10
uavs = []
for i in range(num_uavs):
    uav = dict(generic_uav)
    uav["position"] = np.array([np.random.rand() * 100, np.random.rand() * 100 - 110, 50])
    uavs.append(uav)

# Plotting the positions in 3d
fig = plt.figure()
axis = fig.add_subplot(projection='3d')
axis.scatter([uavs[i]["position"][0] for i in range(num_uavs)],
              [uavs[i]["position"][1] for i in range(num_uavs)],
              [uavs[i]["position"][2] for i in range(num_uavs)],
              color="red")
plt.title("Sample UAV Postions")
plt.show()

In [ ]:
# Defining Generic Ground User

generic_gu = {
    "device_type": "rx",
    "position": np.zeros(3),
    "velocity": np.zeros(3),
    "color": np.array([0, 1, 0]),
    "bandwidth": 50,
    "desired_throughput": 375000
}

# Configuring some sample Ground Users
num_gus = 100
gus = []
for i in range(num_gus):
    gu = dict(generic_gu)
    gu["position"] = np.array([np.random.rand() * 100, np.random.rand() * 100 - 110, 2])
    gus.append(gu)

# Plotting UAV and Ground User Positions
fig = plt.figure()
axis = fig.add_subplot(projection='3d')
axis.scatter([uavs[i]["position"][0] for i in range(num_uavs)],
              [uavs[i]["position"][1] for i in range(num_uavs)],
              [uavs[i]["position"][2] for i in range(num_uavs)],
              color="red")
axis.scatter([gus[i]["position"][0] for i in range(num_gus)],
           [gus[i]["position"][1] for i in range(num_gus)],
           [gus[i]["position"][2] for i in range(num_gus)],
           color="blue")
plt.title("Sample UAV and Ground User Positions")
plt.show()

In [ ]:
# Generating Sample Base Stations

# Defining a generic base station, then randomizing positions
generic_base_station = {
    "device_type": "rx",
    "position": np.zeros(3),
    "color": np.array([0, 0, 1]),
    "bandwidth": 50,
    "signal_power": 10,
    "throughput_capacity": 625000000,
    "battery_capacity": 1000000
}

# Adding some base stations with random positions
num_bs = 3
bss = []
for i in range(num_bs):
    bs = dict(generic_base_station)
    bs["position"] = np.array([np.random.rand() * 100, np.random.rand() * 100 - 110, 75])
    bss.append(bs)

# Plotting UAV, Ground User, and Base Station Positions
fig = plt.figure()
axis = fig.add_subplot(projection='3d')
axis.scatter([uavs[i]["position"][0] for i in range(num_uavs)],
              [uavs[i]["position"][1] for i in range(num_uavs)],
              [uavs[i]["position"][2] for i in range(num_uavs)],
              color="red")
axis.scatter([gus[i]["position"][0] for i in range(num_gus)],
           [gus[i]["position"][1] for i in range(num_gus)],
           [gus[i]["position"][2] for i in range(num_gus)],
           color="blue")
axis.scatter([bss[i]["position"][0] for i in range(num_bs)],
           [bss[i]["position"][1] for i in range(num_bs)],
           [bss[i]["position"][2] for i in range(num_bs)],
           color="green")
plt.title("Sample UAV, Ground User, and Base Station Positions")
plt.show()

In [ ]:
# Creating the scene with the sample devices
lake_wheeler_path = "/home/everetttucker471/Documents/RL-AERPAW-DT/data/aerpaw/LakeWheelerBuildings/lake_wheeler_road_aerpaw.xml"
raleigh_path = '../data/RaleighUnionSquareStandardReference/raleigh_union_square.xml'
env = AERPAWEnv(scene_path=lake_wheeler_path,
                uavs=uavs, ground_users=gus, base_stations=bss, temperature=300)
env.visualize()

In [ ]:
# Computing SNR values between the transmitters and receivers in the scene
print(len(env.getSignalPowers()))

In [ ]:
snr_dict = env.getSNR(max_depth=2, num_samples=100000, sampling_frequency=1.0, )  # Using default arguments, should be alright
print(f"Number of Recievers: {len(snr_dict)}")
print(f"Number of Transmitters: {len(snr_dict["gu0"])}")

In [ ]:
print(snr_dict["gu0"])

## Visualization versus AERPAW SNR Data
* To visualize the effectiveness of this model, we visualize its comparison to AERPAW SNR data generated using a path channel model
* This data is taken from the AERPAW team in the context of the AADM Challenge

In [ ]:
# Reading data
import pandas as pd
import sys
import numpy as np
import matplotlib.pyplot as plt
from AERPAWEnvironment import AERPAWEnv
sys.path.append("..")

def parseTimeToken(token):
    token = token.strip("]")
    comp = token.split(":")
    return int(comp[0]) * 3600 + int(comp[1]) * 60 + int(float(comp[2]))
    
num_aerpaw_bs = 4
snrs = []
for i in range(num_aerpaw_bs):
    snrs.append({})

with open("../data/aerpaw/vehicle_log.txt", "r") as f:
    for line in f.readlines():
        tokens = line.split(" ")
        for i in range(num_aerpaw_bs):
            if f"BS{i + 1}>SNR" in line:
                snrs[i][parseTimeToken(tokens[1])] = float(tokens[-1])

# Converting to pandas Dataframe
df = pd.DataFrame()
df.insert(0, "Time", list(snrs[0].keys()))
for i in range(num_aerpaw_bs):
    df.insert(i + 1, f"LW{i + 1}", list(snrs[i].values()))
print(df)

In [ ]:
# Plotting the SNR values over time for visualization
for i in range(num_aerpaw_bs):
    plt.plot(df["Time"], df[f"LW{i + 1}"], color=np.random.rand(3))
plt.title("AERPAW SNR (dB) vs. Time (seconds)")
plt.xlabel("Time (Seconds)")
plt.ylabel("AERPAW SNR (dB)")
plt.show()

In [ ]:
# Finding the offset to ignore the null snr data
OFFSET = 0
for i in range(100):
    if df["LW1"][i] > 0:
        OFFSET = i
        break
        
print(f"The data really starts at time={OFFSET}")

In [ ]:
import sionna.rt
import numpy as np
import mitsuba as mi
from enum import Enum
from typing import Optional, Final, Tuple, Dict, List
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver
from pyproj import Transformer
from pyproj.enums import TransformDirection

# Position of LW1 in lat/lon/alt - (deg/deg/m)
ORIGIN_LAT_LON: Final[Dict[str, float]] = {"lat": 35.72750947, "lon": -78.69595819, "alt": 82.973}
# XYZ Offset Based on Lake Wheeler Scene - (ft/ft/ft)
SIONNA_OFFSET: Final[List[float]] = [-113, 16, 120]
# SIONNA_OFFSET: Final[List[float]] = [2020.5, 1971.5, 312]


class CoordinateConverter:
    """WGS84 converter between geodetic (lat/lon/alt) and local ENU coordinates."""

    def __init__(self, reference_origin: Optional[Dict[str, float]] = None):
        if not reference_origin:
            reference_origin = ORIGIN_LAT_LON
        self.origin = reference_origin

        pipeline = (
            f"+proj=pipeline "
            f"+step +proj=unitconvert +xy_in=deg +z_in=m +xy_out=rad +z_out=m "
            f"+step +proj=cart +ellps=WGS84 "
            f"+step +proj=topocentric +ellps=WGS84 "                            
            f"+lon_0={self.origin['lon']} +lat_0={self.origin['lat']} +h_0={self.origin['alt']} "
            f"+step +proj=affine +s11=0 +s12=1 +s21=-1 +s22=0 "  # Rotate local coordinates by 180 degrees
            f"+step +proj=unitconvert +xy_in=m +z_in=m +xy_out=m +z_out=m"    
        )

        self.transformer = Transformer.from_pipeline(pipeline)


    def update_reference_origin(self, origin: Dict[str, float]) -> Dict[str, float]:
        self.origin = origin
        pipeline = (
            f"+proj=pipeline "
            f"+step +proj=unitconvert +xy_in=deg +z_in=m +xy_out=rad +z_out=m "
            f"+step +proj=cart +ellps=WGS84 "
            f"+step +proj=topocentric +ellps=WGS84 "                            
            f"+lon_0={self.origin['lon']} +lat_0={self.origin['lat']} +h_0={self.origin['alt']} "
            f"+step +proj=affine +s11=0 +s12=1 +s21=-1 +s22=0 "  # Rotate local coordinates by 180 degrees
            f"+step +proj=unitconvert +xy_in=m +z_in=m +xy_out=m +z_out=m"    
        )

        self.transformer = Transformer.from_pipeline(pipeline)
        return self.origin


    def get_origin(self) -> Dict[str, float]:
        return self.origin


    def lat_lon_alt_to_local(
        self, lat: float, lon: float, alt: float
    ) -> Tuple[float, float, float]:
        """Convert geodetic coordinate to local ENU tuple (x=east, y=north, z=up)."""
        east, north, up = self.transformer.transform(lon, lat, alt, direction=TransformDirection.FORWARD)
        return (east + SIONNA_OFFSET[0], north + SIONNA_OFFSET[1], up + SIONNA_OFFSET[2])


    def local_to_lat_lon_alt(
        self, x: float, y: float, z: float
    ) -> Tuple[float, float, float]:
        lon, lat, alt = self.transformer.transform(x - SIONNA_OFFSET[0],
                                                   y - SIONNA_OFFSET[1],
                                                   z - SIONNA_OFFSET[2], 
                                                   direction=TransformDirection.INVERSE)
        return (lat, lon, alt)


def parseTimeToken(token):
    token = token.strip("]")
    comp = token.split(":")
    return int(comp[0]) * 3600 + int(comp[1]) * 60 + int(float(comp[2]))

### AERPAW Data Inspection
* As you can see from the graph, there is a range before the simulation starts where the SNR is zeroed at -99.0
* Additionally, there are some outlier points where the SNR has a sudden drop
* Other than that, the SNR is fairly consistent over time

In [ ]:
# Grabbing UAV position over time

lat = {}
lon = {}
height = {}
with open("../data/aerpaw/vehicle_out.txt", "r") as f:
    for line in f.readlines():
        tokens = line.split(",")
        time = parseTimeToken(tokens[-3].split(" ")[1])
        lat[time] = float(tokens[2])
        lon[time] = float(tokens[1])
        height[time] = float(tokens[3])
        
lat_list = list(lat.values())
lon_list = list(lon.values())
alt_list = list(height.values())

# # Putting the data into local coordinates, and then into the dataframe

# R = 6371201  # Radius of Earth at Raleigh, in meters
# BASIS = np.array([35.72750947, -78.69595810, 12.973])
# def latLon(lat_lon, basis):
#     """
#     Converts a single point to lat/lon, with respect to the basis point
#     """
#     return np.array([R * np.sin((lat_lon[0] - basis[0]) * np.pi / 180), 
#                     R * np.sin((lat_lon[1] - basis[1]) * np.pi / 180), 
#                     lat_lon[2]])

# # Aligning the coordinates with North / South, so it looks like Google Maps
# ALT_COMP = 25  # The ground level in meters used to adjust the altitude of devices in the scene
# def alignCoordinate(point):
#     return np.array([point[1], point[0], point[2] + ALT_COMP])

# positions = np.array([list(lat.values()), list(lon.values()), list(height.values())]).T
# print(positions.shape)
# for i in range(len(positions)):
#     positions[i] = latLon(positions[i], BASIS)

# df.insert(5, "X", positions[:, 0])
# df.insert(6, "Y", positions[:, 1])
# df.insert(7, "Height", positions[:, 2])
# print(df)
# df.to_csv("aerpaw_experiment_data.csv")

In [ ]:
# Visualizing the location of the UAV over time according to the coordinate conversion we currently have

# Defining the coordinate converter
cc = CoordinateConverter()

# Creating a new scene
# scene = load_scene("/home/everetttucker471/Documents/AERPAW-DT-SIONNA-EXTENSION/data/scenes/lake-wheeler-scene.xml")
scene = load_scene("/home/everetttucker471/Documents/RL-AERPAW-DT/data/aerpaw/LakeWheelerBuildings/lake_wheeler_road_aerpaw.xml")

# Creating a new transmitter object
tx_pos = cc.lat_lon_alt_to_local(ORIGIN_LAT_LON["lat"],
                                 ORIGIN_LAT_LON["lon"],
                                 10)
print("Transmitter Position:")
print(tx_pos)
print()
tx = Transmitter(name="tx", position=tx_pos, velocity=(0, 0, 0))
scene.add(tx)

for i in range(len(lat_list)):
    #device = scene.transmitters["tx"]
    #print(f"Timestep: {i}")
    #print(f"Global Position: {lat_list[i], lon_list[i], alt_list[i]}")
    position = cc.lat_lon_alt_to_local(lat_list[i], 
                                       lon_list[i], 
                                       alt_list[i] - 10)
    #print(f"Local Position: {position}")
    # device.position = mi.Point3f(list(position))
    
    marker = Receiver(name=f"uav_step_{i}", position=list(position))
    scene.add(marker)
    
scene.preview(show_devices=True);

In [ ]:
# Visualizing real-world Trajectory to check for alignment with the Sionna environment

import folium

center_coords = [ORIGIN_LAT_LON["lat"], ORIGIN_LAT_LON["lon"]]

m = folium.Map(location=center_coords, zoom_start=12, tiles="Esri.WorldImagery")

for i in range(len(lat_list)):
    pos = [lat_list[i], lon_list[i]]
    folium.CircleMarker(location=pos, 
                        radius=2,
                        popup=f"pos_{i}",
                        fill=True,
                        fill_color="green",
                        fill_opacity=1).add_to(m)

m.save("lake_wheeler_trajectory.html")

In [ ]:
# Getting positions of base stations
# offsetting for altitude of environment off the ground

BASE_STATIONS_LATLON = np.array([
    [35.72750947, -78.69595810, 12.973],
    [35.72821305, -78.70090823, 8.947],
    [35.72491205, -78.69190014, 2.345],
    [35.73318358, -78.6983642, 19.345]
])

BASE_STATIONS_XYA = []
for bs in BASE_STATIONS_LATLON:
    BASE_STATIONS_XYA.append(alignCoordinate(latLon(bs, BASIS)))
BASE_STATIONS_XYA = np.array(BASE_STATIONS_XYA)
print(BASE_STATIONS_XYA)

In [ ]:
# Map of UAV trajectory and base stations, color-coded by time with lighter as the start

plt.scatter(BASE_STATIONS_XYA[:, 0], BASE_STATIONS_XYA[:, 1], color="red")
plt.scatter(df["X"], df["Y"], c=df["Time"]/max(df["Time"]))
plt.title("UAV Trajectory")
plt.xlabel("X Position (meters)")
plt.ylabel("Y Position (meters)")
plt.show()
# Just need to rotate clockwise 90 degrees to align it with north/south

## Generating Scene
* Now that we have the AERPAW data clearly parsed, we want to generate the same Lake Wheeler environment in the RL-DT framework
* We start by grabbing data from OpenStreetMaps, then instantiate the base stations and single UAV device
* After we have set up the simulated environment, we can move the UAV around the scene in the same pattern, and record the SNR values to each base station

In [ ]:
import tensorflow as tf
print(tf.config.list_physical_devices("GPU"))

In [ ]:
# Instantiating the scene using the minimial AERPAWEnvironment
from AERPAWEnvironment import AERPAWEnv

"""
We use the following material substitutions in the XML file:
<!-- Sionna Material Subsitutions:
	    mat-wall -> itu-concrete
		mat-roof -> itu-plywood
		mat-vegetation -> itu-medium_dry_ground
		mat-roads_residential -> itu-concrete
		mat-roads_service -> itu-concrete
		mat-water -> itu-wet_ground
		mat-roads_primary -> itu-concrete
		mat-paths_footway -> itu-concrete
		mat-roads_track -> itu-concrete
		mat-forest -> itu-medium_dry_ground 
		mat-paths_steps -> itu-concrete
	-->
"""

# Three different degrees of detail, including 100%, 25%, and 4%
# More detail = more memory consumption, and sometimes your kernel will die
terrain_100 = "../data/aerpaw/LakeWheelerTerrain/lake_wheeler_road_aerpaw.xml"
terrain_025 = "../data/aerpaw/LakeWheelerTerrain25/lake_wheeler_road_aerpaw.xml"
terrain_004 = "../data/aerpaw/LakeWheelerTerrain04/lake_wheeler_road_aerpaw.xml"
realistic = "../data/LakeWheelerRealistic/lake_wheeler_road_aerpaw.xml"
buildings = "../data/aerpaw/LakeWheelerBuildings/lake_wheeler_road_aerpaw.xml"

# Test visualization without any scene objects
test_env = AERPAWEnv(scene_path=terrain_004,
                uavs={}, ground_users={}, base_stations={}, temperature=300)
test_env.visualize()

In [ ]:
# We're leaving in only the parameters that are relavent to this experiment, the other ones are left to defaults
BS_SIGNAL_POWER = 2
BS_BANDWIDTH = 3300
UAV_SIGNAL_POWER = 0.5
UAV_BANDWIDTH = 3300

base_stations = [
    {
    "device_type": "tx",
    "position": BASE_STATIONS_XYA[0],
    "color": np.array([0, 1, 0]),
    "bandwidth": BS_BANDWIDTH,
    "signal_power": BS_SIGNAL_POWER},
    {
    "device_type": "tx",
    "position": BASE_STATIONS_XYA[1],
    "color": np.array([0, 1, 0]),
    "bandwidth": BS_BANDWIDTH,
    "signal_power": BS_SIGNAL_POWER},
    {
    "device_type": "tx",
    "position": BASE_STATIONS_XYA[2],
    "color": np.array([0, 1, 0]),
    "bandwidth": BS_BANDWIDTH,
    "signal_power": BS_SIGNAL_POWER},
    {
    "device_type": "tx",
    "position": BASE_STATIONS_XYA[3],
    "color": np.array([0, 1, 0]),
    "bandwidth": BS_BANDWIDTH,
    "signal_power": BS_SIGNAL_POWER
    }
]

uav = [
    {
    "device_type": "rx",
    "position": np.array([df["X"][0], df["Y"][0], df["Height"][0]]),
    "velocity": np.zeros(3),
    "color": np.array([0, 0, 1]),
    "bandwidth": UAV_BANDWIDTH,
    "signal_power": UAV_SIGNAL_POWER
    }
]

env = AERPAWEnv(scene_path=buildings, uavs=uav, ground_users={}, base_stations=base_stations, temperature=300)
env.visualize()

In [ ]:
snrs = env.getSNR(max_depth=1, num_samples=100000, sampling_frequency=1.0, mode="gpu", los=False, reverse=False)

In [ ]:
print(snrs)

In [ ]:
MAX_DEPTH = 1
NUM_SAMPLES = 100000

snr_computed = [[], [], [], []]
plot = False

for i in range(OFFSET, len(df)):
    print(f"Computing {i}")
    position = np.array([df["X"][i], df["Y"][i], df["Height"][i]])
    env.moveAbsUAV(0, position, np.zeros(3))
    
    # Computing and storing the SNR values
    snrs = env.getSNR(max_depth=MAX_DEPTH, num_samples=NUM_SAMPLES, 
                      sampling_frequency=1.0, mode="gpu", los=False, 
                      reverse=False)
    
    # Adding to the arrays
    for j in range(num_aerpaw_bs):
        snr_computed[j].append(snrs["uav0"][f'bs{j}'])
    
    np.savetxt("../data/aerpaw/EnvironmentComparison/snr_computed.csv", np.array(snr_computed), delimiter=',')
    
    if plot:
        # Plotting the current state
        plt.scatter(position[0], position[1], color="green")
        plt.scatter(BASE_STATIONS_XYA[:, 0], BASE_STATIONS_XYA[:, 1], color="red")
        plt.title(f"Environment State at Time {i + 1}")
        plt.xlabel("X Position (meters)")
        plt.ylabel("Y Position (meters)")
        plt.savefig(f"../data/aerpaw/EnvironmentComparison/state{i}.png")

print(np.array(snr_computed).astype(np.float64))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
snr = np.loadtxt("../data/aerpaw/EnvironmentComparison/snr_computed.csv", delimiter=',')
snr = np.maximum(snr, -10)
print(snr)

In [ ]:
computed_colors = ["xkcd:green", "xkcd:blue", "xkcd:red", "xkcd:orange"]
aerpaw_colors = ["xkcd:light green", "xkcd:light blue", "xkcd:light red", "xkcd:light orange"]

for i in range(3, 4):
    plt.plot(snr[i], color=computed_colors[i])
for i in range(3, 4):
    plt.plot(df[f"LW{i + 1}"][OFFSET:].reset_index(drop=True), color=aerpaw_colors[i])
plt.title("AERPAW SNR (dB) vs. Time (seconds) - Low BS, No Buildings")
plt.xlabel("Time (Seconds)")
plt.ylabel("AERPAW SNR (dB)")
plt.show()